# EMBER-Dyn Quickstart 00: Dataset overview and descriptive statistics

**Purpose**: get a first feel for the data: subset composition,
split sizes, label balance, family distribution, time bucket, and
annotation coverage before training any baselines.

**Runtime**: ~30 seconds on Colab free CPU tier.

All figures here are regenerated from the published `manifest_open.csv`
and reproduce Figures 3-5 of the EMBER-Dyn paper.

## 1. Install dependencies and load the manifest

In [ ]:
!pip install -q pandas pyarrow matplotlib

In [ ]:
import pandas as pd
import numpy as np

# EMBER-Dyn open tier is distributed through an author-approved archive. Obtain and verify it first
# (e.g. `pip install zenodo_get && zenodo_get <ZENODO_DOI>`) and point DATA_DIR at the
# extracted dataset folder.
DATA_DIR = 'dataset'  # adjust to your local extraction path
m = pd.read_csv(f'{DATA_DIR}/manifest_open.csv')
print(f'manifest rows: {len(m):,}')
print(f'columns: {len(m.columns)}')
m.head(3)

## 2. Headline statistics

In [ ]:
n_mal = int((m['label'] == 1).sum())
n_ben = int((m['label'] == 0).sum())
n_fam = m.loc[m['label'] == 1, 'family'].dropna().nunique()
print(f'Total samples:      {len(m):,}')
print(f'Malicious:          {n_mal:,}  ({n_mal/len(m)*100:.1f}%)')
print(f'Benign:             {n_ben:,}  ({n_ben/len(m)*100:.1f}%)')
print(f'Distinct families:  {n_fam}')

## 3. Per-subset breakdown (matches Table 2 of the paper)

In [ ]:
subset_counts = m['subset'].value_counts().reindex(
    ['A', 'B', 'C', 'D', 'E1', 'E2', 'E3', 'F', 'Challenge'], fill_value=0)
subset_counts.to_frame('count')   # sums to 10,000 (includes the 483 subset-F samples)

## 4. Train / validation / test / challenge split (matches Methods §3)

In [ ]:
split_counts = m['dyn_split'].value_counts().reindex(
    ['train', 'val', 'test', 'challenge'], fill_value=0)
print(split_counts.to_string())
print('\nlabel balance per split:')
for sp in ['train', 'val', 'test', 'challenge']:
    s = m[m['dyn_split'] == sp]
    print(f'  {sp:>10s}: total={len(s):,}  malicious={(s["label"]==1).sum():,}  benign={(s["label"]==0).sum():,}')

## 5. Subset A: five known families (input to T2 family classifier)

In [ ]:
subset_a = m[m['subset'] == 'A']
subset_a['family'].value_counts().to_frame('count')

## 6. Concept-drift partition (used by T3 time-bucket task)

In [ ]:
tb = m['time_bucket'].value_counts()
print(f"pre-2024-09:  {tb.get('pre', 0):,}")
print(f"post-2024-09: {tb.get('post', 0):,}")
print(f"\nfile type distribution:")
print(m['file_type'].value_counts().to_string())

## 7. Annotation coverage (capa, MITRE ATT&CK, MBC)

Returns the proportion of samples with at least one identifier in each annotation source.

In [ ]:
import ast

def to_list(cell):
    if isinstance(cell, list):
        return cell
    if isinstance(cell, str) and cell.strip() not in ('', '[]', 'None', 'nan'):
        try:
            v = ast.literal_eval(cell)
            return v if isinstance(v, list) else [v]
        except Exception:
            return []
    return []

stats = []
for col in ['behavior', 'packer', 'exploit', 'group', 'caps_list', 'ttps_list', 'mbc_list']:
    sizes = m[col].apply(lambda c: len(to_list(c)))
    non_empty = (sizes > 0).sum()
    stats.append({
        'column': col,
        'non_empty': int(non_empty),
        'coverage_%': round(non_empty / len(m) * 100, 1),
        'mean_size': float(sizes[sizes > 0].mean()) if non_empty > 0 else 0.0,
        'max_size': int(sizes.max()),
    })
pd.DataFrame(stats)

## 8. Detection-ratio distribution (visualises Figure 5 of the paper)

In [ ]:
import matplotlib.pyplot as plt

bins = [-0.001, 0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.001]
labels = ['=0', '(0,0.1)', '[0.1,0.3)', '[0.3,0.5)', '[0.5,0.7)', '[0.7,0.9)', '[0.9,1.0]']
mal = pd.cut(m.loc[m['label'] == 1, 'detection_ratio'], bins=bins, labels=labels).value_counts().reindex(labels, fill_value=0)
ben = pd.cut(m.loc[m['label'] == 0, 'detection_ratio'], bins=bins, labels=labels).value_counts().reindex(labels, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.5))
x = np.arange(len(labels))
ax.bar(x, ben.values, label=f'benign ({ben.sum():,})', color='#9DC3E6')
ax.bar(x, mal.values, bottom=ben.values, label=f'malicious ({mal.sum():,})', color='#1F4E79')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
ax.set_xlabel('Detection-ratio bucket'); ax.set_ylabel('Samples')
ax.legend(frameon=False)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()

## 9. Expected output

Numbers above should match the EMBER-Dyn paper as follows.

| Statistic | Expected |
|---|---|
| Total samples | 10,000 |
| Malicious / benign | 8,126 / 1,874 |
| Distinct AVClass families | 245 |
| Subset A / B / C / D | 5,359 / 1,359 / 381 / 1,845 |
| Subset E1 / E2 / E3 | 109 / 177 / 191 |
| Subset F / Challenge | 483 / 96 |
| (subset total) | 10,000 |
| train / val / test / challenge | 7,000 / 997 / 1,907 / 96 |
| pre / post 2024-09 | 5,427 / 4,573 |
| Subset A families (xmrig/berbew/virlock/cosmu/expiro) | 1,142 / 1,129 / 1,125 / 1,121 / 842 |
| capa coverage | 88.3% |
| MITRE ATT&CK coverage | 82.1% |
| MBC coverage | 85.4% |